# TMDB 5000 Movies: Recommender System & Revenue Prediction

This notebook uses `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` and covers:

1. Load, merge and clean the data
2. Exploratory data analysis
3. **Model A: content-based movie recommender** (TF-IDF + cosine similarity)
4. **Model B: box-office revenue prediction** (regression, trained and compared across 3 algorithms)
5. Saving the trained artifacts

Place both CSV files in the same folder as this notebook (or change `DATA_DIR`).

In [ ]:
import ast, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_DIR = "."   # folder containing the CSVs

## 1. Load and merge the data

In [ ]:
movies  = pd.read_csv(f"{DATA_DIR}/tmdb_5000_movies.csv")
credits = pd.read_csv(f"{DATA_DIR}/tmdb_5000_credits.csv")
print("movies :", movies.shape)
print("credits:", credits.shape)

# credits uses `movie_id`; movies uses `id`. Drop the duplicate title before merging.
df = movies.merge(credits.drop(columns="title"), left_on="id", right_on="movie_id").drop(columns="movie_id")
print("merged :", df.shape)
df.head(3)

In [ ]:
df.isna().sum()[lambda s: s > 0]

### Parse the JSON-like columns
Columns such as `genres`, `keywords`, `cast` and `crew` are stored as stringified lists of dicts.

In [ ]:
def parse(text):
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return []

def names(text, limit=None):
    items = [d["name"] for d in parse(text)]
    return items[:limit] if limit else items

def director(text):
    for d in parse(text):
        if d.get("job") == "Director":
            return d["name"]
    return ""

df["genre_list"]   = df["genres"].apply(names)
df["keyword_list"] = df["keywords"].apply(names)
df["cast_list"]    = df["cast"].apply(lambda x: names(x, 3))      # top 3 billed actors
df["director"]     = df["crew"].apply(director)
df["company_list"] = df["production_companies"].apply(names)

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"]  = df["release_date"].dt.year
df["month"] = df["release_date"].dt.month
df["overview"] = df["overview"].fillna("")

df[["title", "genre_list", "cast_list", "director", "year"]].head()

## 2. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Genre frequency
genre_counts = pd.Series([g for gs in df["genre_list"] for g in gs]).value_counts()
sns.barplot(x=genre_counts.values, y=genre_counts.index, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Movies per genre")

# Movies per decade
df["decade"] = (df["year"] // 10 * 10)
df["decade"].value_counts().sort_index().plot.bar(ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Movies per decade")

# Rating distribution (ignore movies with very few votes)
sns.histplot(df.loc[df["vote_count"] >= 50, "vote_average"], bins=30, ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Vote average (movies with >= 50 votes)")

# Budget vs revenue (log scale), only where both are known
known = df[(df["budget"] > 0) & (df["revenue"] > 0)]
axes[1, 1].scatter(known["budget"], known["revenue"], s=8, alpha=0.4)
axes[1, 1].set_xscale("log"); axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel("Budget"); axes[1, 1].set_ylabel("Revenue"); axes[1, 1].set_title("Budget vs revenue (log-log)")

plt.tight_layout(); plt.show()
print(f"Movies with budget==0: {(df.budget==0).sum()}  |  revenue==0: {(df.revenue==0).sum()}  (missing values encoded as 0)")

In [ ]:
num_cols = ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count", "year"]
plt.figure(figsize=(7, 5))
sns.heatmap(known[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (movies with known budget & revenue)"); plt.show()

## 3. Model A: Content-based recommender

Each movie becomes a "tag soup" of its **overview, genres, keywords, top-3 cast and director**.
We vectorise with **TF-IDF** and recommend the movies with the highest **cosine similarity**.
Multi-word names are collapsed (e.g. `Sam Worthington` → `samworthington`) so people and keywords stay as single tokens.

In [ ]:
def squash(items):
    return [str(i).replace(" ", "").lower() for i in items]

df["tags"] = (
    df["overview"].str.lower() + " "
    + df["genre_list"].apply(lambda x: " ".join(squash(x) * 2)) + " "      # genres weighted x2
    + df["keyword_list"].apply(lambda x: " ".join(squash(x))) + " "
    + df["cast_list"].apply(lambda x: " ".join(squash(x))) + " "
    + df["director"].apply(lambda x: " ".join(squash([x]) * 2) if x else "")  # director weighted x2
)

tfidf = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 1), min_df=2)
tfidf_matrix = tfidf.fit_transform(df["tags"])
print("TF-IDF matrix:", tfidf_matrix.shape)

# TF-IDF rows are L2-normalised, so a linear kernel == cosine similarity
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print("Similarity matrix:", cosine_sim.shape)

In [ ]:
title_to_idx = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()

def recommend(title, n=10):
    key = title.lower()
    if key not in title_to_idx:
        close = [t for t in title_to_idx.index if key in t][:5]
        raise KeyError(f"'{title}' not found. Did you mean: {close}")
    idx = title_to_idx[key]
    scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n + 1]
    out = df.iloc[[i for i, _ in scores]][["title", "genre_list", "director", "vote_average", "year"]].copy()
    out.insert(1, "similarity", [round(s, 3) for _, s in scores])
    return out.reset_index(drop=True)

recommend("The Dark Knight Rises", 8)

In [ ]:
recommend("Toy Story", 8)

In [ ]:
recommend("Avatar", 8)

### Quick sanity evaluation
There are no user ratings in this dataset, so we use a proxy: for each sampled movie, what fraction of its top-10 recommendations share **at least one genre** with it, and how often do they share the **director**? Compared against random picks as a baseline.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample = rng.choice(len(df), 500, replace=False)
genre_sets = df["genre_list"].apply(set).values

def genre_hit_rate(idx_to_recs):
    hits = [np.mean([bool(genre_sets[i] & genre_sets[j]) for j in recs]) for i, recs in idx_to_recs.items()]
    return float(np.mean(hits))

model_recs, random_recs = {}, {}
for i in sample:
    top = np.argsort(-cosine_sim[i])[1:11]
    model_recs[i] = top
    random_recs[i] = rng.choice(len(df), 10, replace=False)

print(f"Genre overlap in top-10  | model: {genre_hit_rate(model_recs):.2%}  | random: {genre_hit_rate(random_recs):.2%}")

## 4. Model B: Predict box-office revenue
We train regression models to predict `revenue` from information known **before release** (budget, runtime, genres, release timing, etc.). `popularity`, `vote_count` and `vote_average` are deliberately excluded because they only exist after a film is out (target leakage).

In [ ]:
reg = df[(df["budget"] > 10_000) & (df["revenue"] > 10_000) & (df["runtime"] > 0) & df["year"].notna()].copy()
print("Rows usable for regression:", len(reg), "of", len(df))

# Multi-hot genres
all_genres = sorted(genre_counts.index)
for g in all_genres:
    reg[f"genre_{g}"] = reg["genre_list"].apply(lambda gs, g=g: int(g in gs))

reg["is_english"]       = (reg["original_language"] == "en").astype(int)
reg["n_companies"]      = reg["company_list"].apply(len)
reg["n_genres"]         = reg["genre_list"].apply(len)
reg["is_summer_or_xmas"] = reg["month"].isin([5, 6, 7, 11, 12]).astype(int)
reg["log_budget"]       = np.log1p(reg["budget"])

feature_cols = (["log_budget", "runtime", "year", "month", "is_english", "n_companies", "n_genres", "is_summer_or_xmas"]
                + [f"genre_{g}" for g in all_genres])
X = reg[feature_cols]
y = np.log1p(reg["revenue"])        # heavy right skew -> model log revenue

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                                  subsample=0.8, random_state=RANDOM_STATE),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    cv_r2 = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "model": name,
        "cv_R2_mean": cv_r2.mean(),
        "test_R2": r2_score(y_test, pred),
        "test_RMSE (log)": np.sqrt(mean_squared_error(y_test, pred)),
        "test_MAE ($)": mean_absolute_error(np.expm1(y_test), np.expm1(pred)),
    })

results = pd.DataFrame(rows).set_index("model").round(3)
results

In [ ]:
best_name = results["test_R2"].idxmax()
best_model = models[best_name]
print("Best model:", best_name)

pred = best_model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(np.expm1(y_test), np.expm1(pred), s=10, alpha=0.5)
lims = [1e4, 3e9]
axes[0].plot(lims, lims, "r--"); axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Actual revenue"); axes[0].set_ylabel("Predicted revenue"); axes[0].set_title(f"{best_name}: actual vs predicted")

resid = y_test - pred
sns.histplot(resid, bins=30, ax=axes[1], color="steelblue")
axes[1].set_title("Residuals (log revenue)")
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance (tree-based best model) or coefficients (Ridge)
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols)
else:
    imp = pd.Series(np.abs(best_model[-1].coef_), index=feature_cols)

imp.sort_values().tail(15).plot.barh(figsize=(8, 5), color="steelblue")
plt.title(f"Top features ({best_name})"); plt.show()

In [ ]:
def predict_revenue(budget, runtime, year, month, genres, language="en", n_companies=2):
    row = {c: 0 for c in feature_cols}
    row.update(log_budget=np.log1p(budget), runtime=runtime, year=year, month=month,
               is_english=int(language == "en"), n_companies=n_companies, n_genres=len(genres),
               is_summer_or_xmas=int(month in [5, 6, 7, 11, 12]))
    for g in genres:
        row[f"genre_{g}"] = 1
    return float(np.expm1(best_model.predict(pd.DataFrame([row])[feature_cols])[0]))

est = predict_revenue(150_000_000, 130, 2015, 6, ["Action", "Adventure", "Science Fiction"])
print(f"Estimated revenue for a $150M summer sci-fi action film: ${est:,.0f}")

## 5. Save the trained artifacts

In [ ]:
joblib.dump({"tfidf": tfidf, "cosine_sim": cosine_sim.astype("float32"),
             "titles": df["title"].tolist()}, "recommender.joblib", compress=3)
joblib.dump({"model": best_model, "features": feature_cols}, "revenue_model.joblib", compress=3)
print("Saved recommender.joblib and revenue_model.joblib")

### Notes & next steps
- **Recommender**: add stemming/lemmatisation, try sentence embeddings, or blend similarity with a weighted-rating score (IMDB formula using `vote_average` and `vote_count`).
- **Revenue model**: accuracy is limited by features available pre-release. Adding director/actor "track record" features (average past revenue) typically gives the biggest gain. Tune hyperparameters with `GridSearchCV`/`RandomizedSearchCV`.